In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score
from scipy.stats import mannwhitneyu
from statsmodels.stats.contingency_tables import mcnemar
import shap

In [3]:
df = pd.read_csv("churnData_Fixed.csv")
leak_cols = ["signup_date", "year_month", "window_id"]
model_features = [c for c in df.columns if c not in leak_cols + ["Churn"]]
N_WINDOWS = df["window_id"].nunique()
 
window0 = df[df["window_id"] == 0]
X_window0 = window0[model_features]
y_window0 = window0["Churn"]
neg_count = (y_window0 == 0).sum()
pos_count = (y_window0 == 1).sum()
scale_pos_weight = neg_count / pos_count
 
u_model = CalibratedClassifierCV(
    XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                  eval_metric="logloss", random_state=42, scale_pos_weight=scale_pos_weight),
    method="sigmoid", cv=5,
)
u_model.fit(X_window0, y_window0)
 
window1 = df[df["window_id"] == 1]
X_window1 = window1[model_features]
y_window1 = window1["Churn"]
u_window1 = u_model.predict_proba(X_window1)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_window1, u_window1)
f1_scores_thresh = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
DECISION_THRESHOLD = thresholds[f1_scores_thresh[:-1].argmax()]
 
shap_base_model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                 eval_metric="logloss", random_state=42, scale_pos_weight=scale_pos_weight)
shap_base_model.fit(X_window0, y_window0)
explainer = shap.TreeExplainer(shap_base_model)
 
shap_values_by_window = {}
for w in sorted(df["window_id"].unique()):
    window_data = df[df["window_id"] == w][model_features]
    shap_values_by_window[w] = pd.DataFrame(explainer.shap_values(window_data), columns=model_features)
 
baseline_shap = shap_values_by_window[0]
baseline_mean = baseline_shap.mean()
baseline_std = baseline_shap.std().replace(0, 1e-10)
 
def rank_biserial_effect_size(n1, n2, u_stat):
    return 1 - (2 * u_stat) / (n1 * n2)
 
effect_size_by_window = {}
for w in sorted(df["window_id"].unique()):
    if w == 0:
        continue
    window_shap = shap_values_by_window[w]
    sizes = {}
    for feature in model_features:
        stat, p_value = mannwhitneyu(baseline_shap[feature], window_shap[feature], alternative="two-sided")
        sizes[feature] = abs(rank_biserial_effect_size(len(baseline_shap), len(window_shap), stat))
    effect_size_by_window[w] = sizes
 
def compute_instance_drift_severity(window_shap_df, feature_weights):
    z_scores = (window_shap_df - baseline_mean) / baseline_std
    weights = pd.Series(feature_weights)[model_features]
    return z_scores.abs().mul(weights, axis=1).sum(axis=1)
 
def normalize_0_1(series):
    span = series.max() - series.min()
    if span == 0:
        return series * 0
    return (series - series.min()) / span
 
def get_window_u_d_error(w):
    window_data = df[df["window_id"] == w]
    X_w = window_data[model_features]
    y_w = window_data["Churn"].values
    u_w = u_model.predict_proba(X_w)[:, 1]
    d_w = normalize_0_1(compute_instance_drift_severity(shap_values_by_window[w], effect_size_by_window[w])).values
    y_pred_w = (u_w >= DECISION_THRESHOLD).astype(int)
    is_error_w = (y_pred_w != y_w).astype(int)
    return u_w, d_w, is_error_w
 
# Re-fit lambda exactly as Phase 4 did (windows 1+2 combined, bounded [0,1])
u1, d1, e1 = get_window_u_d_error(1)
u2, d2, e2 = get_window_u_d_error(2)
 
def deferred_by_score(score, budget):
    n = max(1, int(len(score) * budget))
    flags = np.zeros(len(score), dtype=int)
    flags[np.argsort(-score)[:n]] = 1
    return flags
 
DEFERRAL_BUDGET = 0.15
best_f1, best_lambda = -1, None
for lam in np.arange(0.0, 1.05, 0.05):
    s1 = u1 + d1 + lam * u1 * d1
    s2 = u2 + d2 + lam * u2 * d2
    combined_err = np.concatenate([e1, e2])
    combined_def = np.concatenate([deferred_by_score(s1, DEFERRAL_BUDGET), deferred_by_score(s2, DEFERRAL_BUDGET)])
    f1 = f1_score(combined_err, combined_def, zero_division=0)
    if f1 > best_f1:
        best_f1, best_lambda = f1, lam
LAMBDA = best_lambda
 
print(f"Rebuilt Phase 2-4 pipeline. Decision threshold={DECISION_THRESHOLD:.3f}, fitted lambda={LAMBDA:.2f}")

Rebuilt Phase 2-4 pipeline. Decision threshold=0.130, fitted lambda=0.75


In [4]:
def score_uncertainty_only(u, d):
    return u
 
def score_drift_only(u, d):
    return d
 
def score_additive(u, d):
    return u + d
 
def score_interaction(u, d):
    return u + d + LAMBDA * u * d
 
def deferred_or_gate(u, d, budget):
    # OR-gate: defer whoever ranks highly on EITHER signal. Implemented via
    # min(rank_U, rank_D) as priority, then take the budget's worth of
    # lowest-priority-number customers -- this guarantees an EXACT matched
    # budget (a naive "top-15% by U union top-15% by D" would over- or
    # under-shoot the budget depending on how much the two signals agree,
    # making it an unfair comparison against the budget-capped methods).
    rank_u = pd.Series(u).rank(ascending=False, method="first")
    rank_d = pd.Series(d).rank(ascending=False, method="first")
    priority = np.minimum(rank_u.values, rank_d.values)
    n = max(1, int(len(u) * budget))
    flags = np.zeros(len(u), dtype=int)
    flags[np.argsort(priority)[:n]] = 1
    return flags
 
METHODS = {
    "uncertainty_only": score_uncertainty_only,
    "drift_only": score_drift_only,
    "additive": score_additive,
    "interaction (ours)": score_interaction,
}

In [5]:
HELD_OUT_WINDOWS = [w for w in sorted(df["window_id"].unique()) if w not in (0, 1, 2)]
 
ablation_results = []
for w in HELD_OUT_WINDOWS:
    u_w, d_w, err_w = get_window_u_d_error(w)
 
    for name, score_fn in METHODS.items():
        score = score_fn(u_w, d_w)
        deferred = deferred_by_score(score, DEFERRAL_BUDGET)
        ablation_results.append({
            "window_id": w, "method": name,
            "precision": precision_score(err_w, deferred, zero_division=0),
            "recall": recall_score(err_w, deferred, zero_division=0),
            "f1": f1_score(err_w, deferred, zero_division=0),
        })
 
    deferred_or = deferred_or_gate(u_w, d_w, DEFERRAL_BUDGET)
    ablation_results.append({
        "window_id": w, "method": "or_gate",
        "precision": precision_score(err_w, deferred_or, zero_division=0),
        "recall": recall_score(err_w, deferred_or, zero_division=0),
        "f1": f1_score(err_w, deferred_or, zero_division=0),
    })
 
ablation_df = pd.DataFrame(ablation_results)
 
print(f"\n--- Deferral F1 by method, per held-out window (budget=15%) ---")
pivot_f1 = ablation_df.pivot(index="window_id", columns="method", values="f1")
print(pivot_f1.round(4))
 
print(f"\n--- Average across held-out windows 3-9 ---")
avg_by_method = ablation_df.groupby("method")[["precision", "recall", "f1"]].mean().sort_values("f1", ascending=False)
print(avg_by_method.round(4))
 
ablation_df.to_csv("phase5_ablation_results.csv", index=False)
print("\nSaved phase5_ablation_results.csv")


--- Deferral F1 by method, per held-out window (budget=15%) ---
method     additive  drift_only  interaction (ours)  or_gate  uncertainty_only
window_id                                                                     
3            0.3288      0.1849              0.3562   0.2603            0.4589
4            0.3168      0.2174              0.3292   0.2919            0.3913
5            0.2966      0.2069              0.3241   0.2966            0.3172
6            0.3032      0.2387              0.3032   0.2903            0.2839
7            0.2132      0.2320              0.1881   0.1881            0.2132
8            0.1649      0.2340              0.1436   0.1489            0.1436
9            0.1452      0.2155              0.1358   0.1639            0.0843

--- Average across held-out windows 3-9 ---
                    precision  recall      f1
method                                       
uncertainty_only       0.4068  0.2038  0.2703
interaction (ours)     0.3878  0.1908  0.

In [6]:
BUDGET_SWEEP = np.arange(0.05, 0.55, 0.05)
 
pooled_u, pooled_d, pooled_err = [], [], []
for w in HELD_OUT_WINDOWS:
    u_w, d_w, err_w = get_window_u_d_error(w)
    pooled_u.append(u_w)
    pooled_d.append(d_w)
    pooled_err.append(err_w)
pooled_u = np.concatenate(pooled_u)
pooled_d = np.concatenate(pooled_d)
pooled_err = np.concatenate(pooled_err)
 
risk_coverage_results = []
for budget in BUDGET_SWEEP:
    coverage = 1 - budget
    for name, score_fn in METHODS.items():
        score = score_fn(pooled_u, pooled_d)
        deferred = deferred_by_score(score, budget)
        accepted_mask = deferred == 0  # auto-processed, NOT deferred
        risk = pooled_err[accepted_mask].mean() if accepted_mask.sum() > 0 else np.nan
        risk_coverage_results.append({"budget": budget, "coverage": coverage, "method": name, "risk": risk})
 
    deferred_or = deferred_or_gate(pooled_u, pooled_d, budget)
    accepted_mask_or = deferred_or == 0
    risk_or = pooled_err[accepted_mask_or].mean() if accepted_mask_or.sum() > 0 else np.nan
    risk_coverage_results.append({"budget": budget, "coverage": coverage, "method": "or_gate", "risk": risk_or})
 
risk_coverage_df = pd.DataFrame(risk_coverage_results)
risk_coverage_df.to_csv("phase5_risk_coverage.csv", index=False)
print("\n--- Risk-coverage table (pooled held-out windows) saved to phase5_risk_coverage.csv ---")
print(risk_coverage_df.pivot(index="coverage", columns="method", values="risk").round(4))
 


--- Risk-coverage table (pooled held-out windows) saved to phase5_risk_coverage.csv ---
method    additive  drift_only  interaction (ours)  or_gate  uncertainty_only
coverage                                                                     
0.50        0.2491      0.2901              0.2438   0.2462            0.2122
0.55        0.2581      0.2954              0.2496   0.2518            0.2242
0.60        0.2711      0.2989              0.2671   0.2650            0.2360
0.65        0.2836      0.3036              0.2789   0.2830            0.2530
0.70        0.2932      0.3092              0.2898   0.2947            0.2651
0.75        0.3004      0.3115              0.2980   0.3037            0.2785
0.80        0.3093      0.3152              0.3076   0.3111            0.2974
0.85        0.3154      0.3207              0.3142   0.3176            0.3085
0.90        0.3191      0.3239              0.3162   0.3218            0.3171
0.95        0.3230      0.3239              0.3234   

In [7]:
N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)
n_pooled = len(pooled_u)
 
bootstrap_results = {name: [] for name in METHODS if name != "interaction (ours)"}
bootstrap_results["or_gate"] = []
 
for i in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_pooled, n_pooled)
    u_b, d_b, err_b = pooled_u[idx], pooled_d[idx], pooled_err[idx]
 
    f1_interaction = f1_score(err_b, deferred_by_score(score_interaction(u_b, d_b), DEFERRAL_BUDGET), zero_division=0)
 
    for name in bootstrap_results:
        if name == "or_gate":
            deferred_base = deferred_or_gate(u_b, d_b, DEFERRAL_BUDGET)
        else:
            deferred_base = deferred_by_score(METHODS[name](u_b, d_b), DEFERRAL_BUDGET)
        f1_base = f1_score(err_b, deferred_base, zero_division=0)
        bootstrap_results[name].append(f1_interaction - f1_base)
 
print(f"\n--- Bootstrap 95% CI: F1(interaction) - F1(baseline), {N_BOOTSTRAP} resamples, pooled held-out windows ---")
for name, diffs in bootstrap_results.items():
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    mean_diff = diffs.mean()
    significant = lo > 0 or hi < 0  # CI excludes zero
    print(f"vs {name:20s}: mean diff={mean_diff:+.4f}, 95% CI=[{lo:+.4f}, {hi:+.4f}], "
          f"significant={significant}")
 


--- Bootstrap 95% CI: F1(interaction) - F1(baseline), 1000 resamples, pooled held-out windows ---
vs uncertainty_only    : mean diff=-0.0205, 95% CI=[-0.0409, -0.0008], significant=True
vs drift_only          : mean diff=+0.0230, 95% CI=[+0.0026, +0.0441], significant=True
vs additive            : mean diff=+0.0033, 95% CI=[-0.0068, +0.0128], significant=False
vs or_gate             : mean diff=+0.0127, 95% CI=[-0.0044, +0.0331], significant=False


In [8]:
error_mask = pooled_err == 1
u_err, d_err, err_err = pooled_u[error_mask], pooled_d[error_mask], pooled_err[error_mask]
 
deferred_interaction_err = deferred_by_score(score_interaction(pooled_u, pooled_d), DEFERRAL_BUDGET)[error_mask]
 
print(f"\n--- McNemar's test: interaction score vs each baseline, on {error_mask.sum()} actual error-cases (pooled) ---")
for name, score_fn in METHODS.items():
    if name == "interaction (ours)":
        continue
    deferred_base_err = deferred_by_score(score_fn(pooled_u, pooled_d), DEFERRAL_BUDGET)[error_mask]
 
    both = ((deferred_interaction_err == 1) & (deferred_base_err == 1)).sum()
    only_interaction = ((deferred_interaction_err == 1) & (deferred_base_err == 0)).sum()
    only_base = ((deferred_interaction_err == 0) & (deferred_base_err == 1)).sum()
    neither = ((deferred_interaction_err == 0) & (deferred_base_err == 0)).sum()
 
    table = [[both, only_interaction], [only_base, neither]]
    result = mcnemar(table, exact=True if (only_interaction + only_base) < 25 else False)
    print(f"vs {name:20s}: caught-by-both={both}, only-interaction={only_interaction}, "
          f"only-baseline={only_base}, neither={neither}  ->  p-value={result.pvalue:.4f}")
 
deferred_or_err = deferred_or_gate(pooled_u, pooled_d, DEFERRAL_BUDGET)[error_mask]
both = ((deferred_interaction_err == 1) & (deferred_or_err == 1)).sum()
only_interaction = ((deferred_interaction_err == 1) & (deferred_or_err == 0)).sum()
only_base = ((deferred_interaction_err == 0) & (deferred_or_err == 1)).sum()
neither = ((deferred_interaction_err == 0) & (deferred_or_err == 0)).sum()
table = [[both, only_interaction], [only_base, neither]]
result = mcnemar(table, exact=True if (only_interaction + only_base) < 25 else False)
print(f"vs {'or_gate':20s}: caught-by-both={both}, only-interaction={only_interaction}, "
      f"only-baseline={only_base}, neither={neither}  ->  p-value={result.pvalue:.4f}")
 
 
print("\nPhase 5 complete.")
print("`ablation_df`       -> per-window, per-method precision/recall/F1 (also saved to CSV)")
print("`risk_coverage_df`  -> risk vs coverage sweep, all methods (also saved to CSV)")
print("`bootstrap_results` -> F1(interaction)-F1(baseline) distributions, 1000 resamples each")
print("`LAMBDA`            -> re-fitted here identically to Phase 4, for standalone reproducibility")


--- McNemar's test: interaction score vs each baseline, on 1601 actual error-cases (pooled) ---
vs uncertainty_only    : caught-by-both=179, only-interaction=105, only-baseline=129, neither=1188  ->  p-value=0.1327
vs drift_only          : caught-by-both=149, only-interaction=135, only-baseline=108, neither=1209  ->  p-value=0.0953
vs additive            : caught-by-both=257, only-interaction=27, only-baseline=22, neither=1295  ->  p-value=0.5677
vs or_gate             : caught-by-both=197, only-interaction=87, only-baseline=73, neither=1244  ->  p-value=0.3041

Phase 5 complete.
`ablation_df`       -> per-window, per-method precision/recall/F1 (also saved to CSV)
`risk_coverage_df`  -> risk vs coverage sweep, all methods (also saved to CSV)
`bootstrap_results` -> F1(interaction)-F1(baseline) distributions, 1000 resamples each
`LAMBDA`            -> re-fitted here identically to Phase 4, for standalone reproducibility
